In [ ]:
import os
import rasterio as rio
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.path as mpth
from pathlib import Path

import Functions
import importlib

importlib.reload(Functions)

output_folder = Path('/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents')
app_path = Functions.get_input_path() / 'App'
input_path = app_path / 'Documents' / 'csv'

In [ ]:
sigma_TM = pd.DataFrame(pd.read_csv(input_path / 'TimeSeries_sigma.csv'))
sigma_TM['Date'] = pd.to_datetime(sigma_TM['Date'])
sigma_TM.drop(sigma_TM[sigma_TM['mean'] == 0].index,inplace=True)

sigma_TM = sigma_TM.sort_values(by=['Code', 'Band', 'Date'])

display(sigma_TM)

In [ ]:
phen = pd.read_csv('/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents/csv/Averaged_Phenology_Stage.csv')
df_long = phen.melt(
    id_vars="Code",        # la colonna che resta fissa
    var_name="Date",       # nuova colonna con i nomi delle vecchie colonne (le date)
    value_name="Phen"      # nuova colonna con i valori
)
df_long["Date"] = pd.to_datetime(df_long["Date"], format="%Y-%m-%d %H:%M:%S").dt.normalize()
df_long.dropna(subset='Phen', inplace=True)
display(df_long[df_long['Code'] == 1553694])
phen_PCHIP = []

for c in df_long['Code'].unique():

    df = df_long[df_long['Code'] == c].copy()
    df = df.set_index('Date')                    # <-- Date diventa l'indice
    df = df.sort_index()                         # utile per l'interpolazione temporale

    raggio_date_giornaliero = pd.date_range(start=df.index.min(), end=df.index.max(), freq='D')
    df = df.reindex(raggio_date_giornaliero)
    df.reset_index(names='Date', inplace=True)   # ora 'Date' non esiste più come colonna: nessun conflitto
    df['Code'] = c
    n_punti = df['Phen'].notna().sum()

    if n_punti >= 2:
        df['Phen_PCHIP'] = df['Phen'].interpolate(method='pchip')
    else:
        # troppo pochi punti: niente interpolazione, si tiene il valore osservato
        df['Phen_PCHIP'] = df['Phen']
        print(f"Code {c}: solo {n_punti} punto/i osservato/i, PCHIP saltata")
#Piecewise Cubic Hermite Interpolating Polynomial (PCHIP)

    phen_PCHIP.append(df)

phen_generale = pd.concat(phen_PCHIP, ignore_index=True)

phen_generale = phen_generale[['Date', 'Code', 'Phen', 'Phen_PCHIP']]
phen_generale.sort_values(by=['Code', 'Date'], inplace=True, ignore_index=True)
display(phen_generale[phen_generale['Code'] == 1553694])



In [ ]:
mapping_df = sigma_TM[['Code', 'Type']].drop_duplicates()
df2_updated = pd.merge(phen_generale, mapping_df, on='Code', how='left')
display(df2_updated)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Assicurati che 'Date' sia in formato datetime
df2_updated['Date'] = pd.to_datetime(df2_updated['Date'])

# Creazione del grafico a linee
plt.figure(figsize=(12, 6))

# Usiamo sns.lineplot invece di scatterplot
sns.lineplot(
    data=df2_updated, 
    x='Date', 
    y='Phen_PCHIP', 
    hue='Type',       # Colori diversi per ogni tipo
    style='Type',     # (Opzionale) Cambia anche lo stile della linea (es. tratteggiata)
    markers=True,     # (Opzionale) Aggiunge un punto dove sono presenti dati reali
    dashes=False ,
    errorbar='sd'     # (Opzionale) Forza linee continue
)


plt.title('Timeseries of phenology stage')
plt.xlabel('Date')
plt.ylabel('Phenology Stage')
plt.grid(True, linestyle='--', alpha=0.6)
plt.savefig('TS_phen', dpi=600)
plt.show()

In [ ]:
df_rainfall = pd.read_csv(r'/home/frank/Desktop/f.chiapperino_local/VALENCIA/SCUOLA/App/Documents//csv/rainfall_total.csv', index_col=0)
df_rainfall.index = pd.to_datetime(df_rainfall.index, format='%Y-%m-%d')
prima_corrispondenza = '2017-01-04'

print(f"Data trovata: {prima_corrispondenza}")
df_filtered = df_rainfall.loc[prima_corrispondenza:]


df_weekly = df_filtered.resample('6D', label='left').sum()
display(df_weekly)
